# Evaluating Agent Outputs: Practice Exercise

In this exercise, you'll implement two evaluation techniques for agent outputs:

1. **Semantic Matching** - Compare agent outputs based on meaning rather than exact text
2. **LLM-as-a-Judge for Outputs** - Use an LLM to evaluate output quality against criteria

These techniques complement trajectory evaluation by focusing on **what** the agent produces rather than **how** it got there.

**What you'll implement:**
- A semantic similarity evaluator using embeddings
- An LLM-as-a-judge evaluator for output quality

**Estimated time:** 20 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [ ]:
# Setup - run this cell first

from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
import numpy as np

print("Setup complete!")

## The Recipe Agent

We'll evaluate a recipe lookup agent that helps users find cooking information.

In [ ]:
# Mock recipe database
RECIPE_DATABASE = {
    "pasta carbonara": {
        "name": "Pasta Carbonara",
        "ingredients": ["spaghetti", "eggs", "parmesan cheese", "pancetta", "black pepper"],
        "time": "25 minutes",
        "difficulty": "Medium",
        "instructions": "Cook pasta. Fry pancetta. Mix eggs with cheese. Combine all with pasta water."
    },
    "chicken stir fry": {
        "name": "Chicken Stir Fry",
        "ingredients": ["chicken breast", "bell peppers", "soy sauce", "garlic", "ginger", "vegetable oil"],
        "time": "20 minutes",
        "difficulty": "Easy",
        "instructions": "Slice chicken and vegetables. Stir fry chicken first, then add vegetables. Season with soy sauce."
    },
    "chocolate cake": {
        "name": "Chocolate Cake",
        "ingredients": ["flour", "cocoa powder", "sugar", "eggs", "butter", "baking powder", "milk"],
        "time": "60 minutes",
        "difficulty": "Medium",
        "instructions": "Mix dry ingredients. Combine wet ingredients. Fold together. Bake at 350F for 35 minutes."
    }
}


@tool
def lookup_recipe(dish_name: str) -> str:
    """Look up a recipe by dish name.
    
    Args:
        dish_name: The name of the dish to look up
    
    Returns:
        Recipe information or a not found message
    """
    dish_key = dish_name.lower().strip()
    
    if dish_key in RECIPE_DATABASE:
        recipe = RECIPE_DATABASE[dish_key]
        return (
            f"Recipe: {recipe['name']}\n"
            f"Ingredients: {', '.join(recipe['ingredients'])}\n"
            f"Time: {recipe['time']}\n"
            f"Difficulty: {recipe['difficulty']}\n"
            f"Instructions: {recipe['instructions']}"
        )
    
    return f"Recipe for '{dish_name}' not found in database."


# Create the agent
agent_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
recipe_agent = create_agent(model=agent_model, tools=[lookup_recipe])

print("Recipe agent created!")
print(f"Available recipes: {', '.join(r['name'] for r in RECIPE_DATABASE.values())}")

## Generate Some Agent Outputs

Let's run the agent on a few queries to generate outputs we can evaluate.

In [ ]:
def get_agent_response(agent, query: str) -> str:
    """Run the agent and return only the final response text."""
    result = agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content


# Test queries and collect responses
test_cases = [
    {
        "query": "What ingredients do I need for pasta carbonara?",
        "expected": "You need spaghetti, eggs, parmesan cheese, pancetta, and black pepper for pasta carbonara."
    },
    {
        "query": "How long does chicken stir fry take to make?",
        "expected": "Chicken stir fry takes about 20 minutes to prepare."
    },
    {
        "query": "Is chocolate cake easy to make?",
        "expected": "Chocolate cake has a medium difficulty level."
    }
]

# Get agent responses
for case in test_cases:
    case["actual"] = get_agent_response(recipe_agent, case["query"])
    print(f"Query: {case['query']}")
    print(f"Response: {case['actual']}\n")

## Part 1: Semantic Matching with Embeddings

Exact string matching is too rigid for evaluating agent outputs. Two responses can have the same meaning but different wording:

- "It takes 20 minutes"
- "The preparation time is about 20 min"

**Semantic matching** uses embeddings to compare the meaning of texts rather than their exact characters.

### How it works:
1. Convert both texts to embedding vectors using an embedding model
2. Calculate cosine similarity between the vectors
3. If similarity exceeds a threshold (e.g., 0.85), consider them a match

### Your Task:
Implement a semantic similarity evaluator that compares an agent's actual output against an expected output.

In [ ]:
def cosine_similarity(vec1: list[float], vec2: list[float]) -> float:
    """Calculate cosine similarity between two vectors.
    
    Cosine similarity = (A . B) / (||A|| * ||B||)
    
    Args:
        vec1: First embedding vector
        vec2: Second embedding vector
    
    Returns:
        Similarity score between -1 and 1 (1 = identical, 0 = orthogonal)
    """
    a = np.array(vec1)
    b = np.array(vec2)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# This helper is provided for you
print("Cosine similarity function ready!")

In [ ]:
def create_semantic_evaluator(threshold: float = 0.85):
    """Create a semantic similarity evaluator using OpenAI embeddings.
    
    Args:
        threshold: Minimum similarity score to consider a match (0.0 to 1.0)
    
    Returns:
        A function that takes (actual, expected) strings and returns an evaluation dict
    """
    # TODO 1: Create an OpenAIEmbeddings instance
    # Use model="text-embedding-3-small" for efficiency
    embeddings = None  # Replace with OpenAIEmbeddings instance
    
    def evaluate(actual: str, expected: str) -> dict:
        """Evaluate semantic similarity between actual and expected outputs.
        
        Args:
            actual: The agent's actual output
            expected: The expected/reference output
        
        Returns:
            Dict with 'score' (bool), 'similarity' (float), and 'threshold' (float)
        """
        # TODO 2: Get embeddings for both texts
        # Use embeddings.embed_documents([text]) to get embedding vectors
        # This returns a list of vectors, so take the first element [0]
        actual_embedding = None  # Get embedding for actual
        expected_embedding = None  # Get embedding for expected
        
        # TODO 3: Calculate cosine similarity between the embeddings
        similarity = None  # Use cosine_similarity function
        
        # TODO 4: Return evaluation result
        # score should be True if similarity >= threshold
        return {
            "score": None,  # Boolean: did it pass?
            "similarity": None,  # Float: the actual similarity score
            "threshold": threshold
        }
    
    return evaluate

## Test Semantic Matching

Run your semantic evaluator on the test cases.

In [ ]:
# Create the evaluator
semantic_eval = create_semantic_evaluator(threshold=0.80)

# Test on our cases
print("=== Semantic Matching Results ===")
print()

for case in test_cases:
    result = semantic_eval(case["actual"], case["expected"])
    
    print(f"Query: {case['query']}")
    print(f"Expected: {case['expected']}")
    print(f"Actual: {case['actual'][:100]}..." if len(case['actual']) > 100 else f"Actual: {case['actual']}")
    print(f"Similarity: {result['similarity']:.3f}")
    print(f"Pass: {result['score']}")
    print()

## Part 2: LLM-as-a-Judge for Output Quality

While semantic matching checks if outputs are similar, **LLM-as-a-Judge** can evaluate outputs against specific criteria:

- Is the response **accurate** based on the source data?
- Is the response **complete** - does it answer the full question?
- Is the response **concise** - no unnecessary information?
- Is the response **well-formatted** and easy to read?

### Your Task:
Implement an LLM-as-a-Judge evaluator that assesses output quality using custom criteria.

In [ ]:
def create_output_judge(criteria: str):
    """Create an LLM-as-a-Judge evaluator for agent outputs.
    
    Args:
        criteria: The evaluation criteria as a string describing what to check
    
    Returns:
        A function that takes (query, output) and returns an evaluation dict
    """
    # TODO 1: Create a ChatOpenAI instance for the judge
    # Use gpt-4o-mini with temperature=0 for consistent evaluation
    judge_model = None  # Replace with ChatOpenAI instance
    
    def evaluate(query: str, output: str) -> dict:
        """Evaluate an agent output using LLM-as-a-Judge.
        
        Args:
            query: The original user query
            output: The agent's output to evaluate
        
        Returns:
            Dict with 'score' (bool), 'reasoning' (str)
        """
        # TODO 2: Create the evaluation prompt
        # The prompt should:
        # - Present the criteria to evaluate against
        # - Show the user's query
        # - Show the agent's output
        # - Ask for a PASS or FAIL verdict with reasoning
        # - Instruct the model to respond in a specific format
        eval_prompt = f"""You are an evaluation judge. Assess the following agent output.

EVALUATION CRITERIA:
{criteria}

USER QUERY:
{query}

AGENT OUTPUT:
{output}

Evaluate whether the output meets the criteria. Respond in this exact format:
VERDICT: [PASS or FAIL]
REASONING: [Your explanation]
"""
        
        # TODO 3: Call the judge model with the prompt
        # Use judge_model.invoke() with a messages list
        response = None  # Get the judge's response
        
        # TODO 4: Parse the response to extract verdict and reasoning
        # The response content will contain "VERDICT: PASS" or "VERDICT: FAIL"
        response_text = None  # Get .content from response
        
        # Parse verdict (check if "VERDICT: PASS" is in the response)
        score = None  # Boolean based on verdict
        
        # Extract reasoning (everything after "REASONING:")
        reasoning = None  # Extract the reasoning text
        
        return {
            "score": score,
            "reasoning": reasoning
        }
    
    return evaluate

## Test LLM-as-a-Judge

Create an evaluator with specific criteria and test it on the agent outputs.

In [ ]:
# Define evaluation criteria
quality_criteria = """The response should:
1. Directly answer the user's question
2. Be accurate based on recipe information
3. Be concise (not overly verbose)
4. Not include information the user didn't ask for
"""

# Create the judge
output_judge = create_output_judge(quality_criteria)

# Test on our cases
print("=== LLM-as-a-Judge Results ===")
print()

for case in test_cases:
    result = output_judge(case["query"], case["actual"])
    
    print(f"Query: {case['query']}")
    print(f"Output: {case['actual'][:100]}..." if len(case['actual']) > 100 else f"Output: {case['actual']}")
    print(f"Pass: {result['score']}")
    print(f"Reasoning: {result['reasoning']}")
    print()

## Combine Both Evaluators

In practice, you might want to use both techniques together. Run this cell to see a combined evaluation.

In [ ]:
def run_combined_evaluation(query: str, actual: str, expected: str):
    """Run both semantic matching and LLM-as-a-Judge evaluation."""
    print(f"Query: {query}")
    print(f"Output: {actual}")
    print()
    
    # Semantic evaluation
    sem_result = semantic_eval(actual, expected)
    print(f"Semantic Match: {'PASS' if sem_result['score'] else 'FAIL'} (similarity: {sem_result['similarity']:.3f})")
    
    # LLM-as-a-Judge evaluation
    judge_result = output_judge(query, actual)
    print(f"Quality Judge: {'PASS' if judge_result['score'] else 'FAIL'}")
    print(f"  Reasoning: {judge_result['reasoning']}")
    
    # Overall pass requires both
    overall = sem_result['score'] and judge_result['score']
    print(f"\nOverall: {'PASS' if overall else 'FAIL'}")
    print("-" * 60)


# Run combined evaluation on all test cases
print("=== Combined Evaluation ===")
print()
for case in test_cases:
    run_combined_evaluation(case["query"], case["actual"], case["expected"])

## Verify Your Solution

Your implementation is successful if:

**Semantic Matching:**
- Returns similarity scores between 0 and 1
- Similar responses (same meaning, different words) get high scores (> 0.8)
- Completely different responses get low scores (< 0.5)

**LLM-as-a-Judge:**
- Returns PASS for outputs that meet the criteria
- Returns FAIL for outputs that don't meet the criteria
- Provides clear reasoning for the verdict

### Key Takeaways

| Technique | Best For | Limitations |
|-----------|----------|-------------|
| Semantic Matching | Checking if output conveys expected information | Doesn't assess quality, just similarity |
| LLM-as-a-Judge | Evaluating nuanced criteria (quality, tone, completeness) | More expensive, can be inconsistent |